In [55]:
import json 
import os  

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "cacchione2014apes")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "caccione_2014_apes_RawData_Collection.csv")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [56]:
import pandas as pd
import numpy as np
import pyreadstat

df = pd.read_csv(complete_path_1)
df['study_id']="cacchione2014apes"
df.columns = map(str.lower, df.columns)
df=df.applymap(lambda s: s.lower() if type(s) == str else s)


In [57]:
df[['month','day', 'year']] = df['date'].str.split('/',expand=True)

In [58]:
df = df.rename(columns={"subject": "ape",
    "response side": "response_side",
    "species": "species_original",
    "condition":"condition_temp"})


In [59]:
comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)
df['ape'] = df['ape'].str.rstrip()
for x,y in zip(df_name['wrong'],df_name['right']):
    df['ape'].replace(x, y, inplace=True)

comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)   
df= df.merge(apedf,left_on='ape', right_on='name', how='left') 
# df.columns
df.rename(columns={"ape": "participant"}, inplace=True)

In [60]:
temp_var = ''
out_list = []
for index, row in df.iterrows():
    if not pd.isna(row['condition_temp']):
        temp_var = row['condition_temp']
    out_list.append(temp_var)
df = df.assign(condition=out_list)

In [61]:
experiment=[]
for index, row in df.iterrows():
    if "5" in str(row['condition']):
        experiment.append('2')
    else:
        experiment.append('1')
df = df.assign(experiment=experiment)

df['condition'].replace('-', '_', inplace=True, regex=True)

condition_list = [['plp','sols'],['lp','slis']]
for x,y in condition_list:
    df['condition'].replace(x, y, inplace=True, regex=True)

df['condition'].replace('obj11', 'obj_11', inplace=True, regex=True)

df['correct'].replace(1, 'incorrect', inplace=True, regex=True)
df['correct'].replace(2, 'correct', inplace=True, regex=True)

df['response_side'].replace('l', 'left', inplace=True, regex=True)
df['response_side'].replace('r', 'right', inplace=True, regex=True)
# df['correct'].unique()

In [62]:
df=df[['study_id', 'experiment','year', 'month', 'day',   'participant', 'sex', 'species',
       'session',  'trial','condition',
       'response_side', 'correct' ]]


In [63]:
for index in range(1,3):
    exp = df[df['experiment'] == str(index)]
    exp = exp.dropna(axis=1, how='all')
    comp_out_path = os.path.join(out_pathway, 'cacchione2014apes_exp'+str(index)+'_standardized.csv')
    exp.to_csv(comp_out_path, encoding='utf-8-sig', index=False)
    names = exp.columns.tolist()
    exp_g = pd.DataFrame(names)
    exp_g = exp_g.rename(columns={0: "column_name"})
    exp_g["description"] = ""
    exp_g=exp_g[["column_name", "description"]]
    comp_out_path_glossary = os.path.join(out_pathway, 'cacchione2014apes_exp'+str(index)+'_glossary.csv')
    exp_g.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)